In [1]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import xml.etree.ElementTree as ET
import glob
from torch.utils.data import DataLoader
import torchvision.transforms as T

In [2]:
class VOCDataset(Dataset):
    def __init__(self, root, split="train", transforms=None):
        """
        root: carpeta raíz del dataset en formato VOC
        split: "train" o "valid"
        transforms: transforms que se aplicarán solo a la imagen
        """
        self.root = root
        self.split = split
        self.transforms = transforms

        self.images = sorted(glob.glob(os.path.join(root, split, "*.jpg")))
        self.annotations = sorted(glob.glob(os.path.join(root, split, "*.xml")))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        xml_path = self.annotations[idx]

        img = Image.open(img_path).convert("RGB")

        # Leer anotación XML VOC
        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            label = obj.find("name").text
            bbox = obj.find("bndbox")

            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(int(label))  # Asegúrate que tus labels sean enteros

        # Convertir a tensores
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }

        # Aplicar solo transform a la imagen
        if self.transforms:
            img = self.transforms(img)

        return img, target

In [3]:
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
])

In [4]:
train_dataset = VOCDataset("coco128.v1i.voc", "train", transforms=transform)
val_dataset   = VOCDataset("coco128.v1i.voc", "valid", transforms=transform)

In [5]:
import torchvision
from torchvision.models.detection.ssd import SSDClassificationHead

# 1. Cargar modelo preentrenado
model = torchvision.models.detection.ssd300_vgg16(weights="DEFAULT")

# 2. Definir número de clases (incluye background)
num_classes = 73

# 3. Reconstruir correctamente la cabeza de clasificación
in_channels = model.backbone.out_channels
num_anchors = model.anchor_generator.num_anchors_per_location()

model.head.classification_head = SSDClassificationHead(
    in_channels,
    num_anchors,
    num_classes
)


AttributeError: 'SSDFeatureExtractorVGG' object has no attribute 'out_channels'

In [6]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

In [7]:
optimizer = torch.optim.SGD(model.parameters(), lr=5e-5, momentum=0.9, weight_decay=5e-4)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 60

for epoch in range(num_epochs):
    model.train()
    for images, targets in train_loader:
        images = [img.to(device) for img in images]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {losses.item():.4f}")


Epoch 1/60 - Loss: 25.5058
Epoch 2/60 - Loss: 17.6267
Epoch 3/60 - Loss: 16.6926
Epoch 4/60 - Loss: 22.1031
Epoch 5/60 - Loss: 24.0411
Epoch 6/60 - Loss: 18.8474
Epoch 7/60 - Loss: 24.8258
Epoch 8/60 - Loss: 12.0057
Epoch 9/60 - Loss: 40.6467
Epoch 10/60 - Loss: 16.9149
Epoch 11/60 - Loss: 33.7914
Epoch 12/60 - Loss: 17.7713
Epoch 13/60 - Loss: 12.1366
Epoch 14/60 - Loss: 210.8894
Epoch 15/60 - Loss: 33.5456
Epoch 16/60 - Loss: 17.6196
Epoch 17/60 - Loss: 24.0158
Epoch 18/60 - Loss: 48.4036
Epoch 19/60 - Loss: 16.8327
Epoch 20/60 - Loss: 27.1165


KeyboardInterrupt: 

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model.to(device)
model.eval()


Using device: cpu


SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

In [ ]:
import torchvision.transforms as T

transforms = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
])


In [16]:
dataset = VOCDataset(
    img_dir="coco128.v1i.voc/images",
    ann_dir="ruta/a/annotations",
    transforms=transforms
)

TypeError: VOCDataset.__init__() got an unexpected keyword argument 'img_dir'

In [10]:
import time
import torch

model.eval()
times = []

with torch.no_grad():
    for images, _ in val_loader:
        images = [img.to(device) for img in images]

        start = time.time()
        outputs = model(images)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        times.append(time.time() - start)

fps = 1 / (sum(times) / len(times))
print(f"FPS: {fps:.2f}")


FPS: 0.57


In [11]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
!pip install pycocotools

In [12]:
import pycocotools
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision()
print("MAP funcionando")


MAP funcionando


In [13]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
metric = MeanAveragePrecision(
    iou_thresholds=[0.5],   # para AP@0.5
    box_format="xyxy"
)

In [15]:
metric.reset()

with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(device) for img in images]

        outputs = model(images)

        preds = []
        gts = []

        for i in range(len(images)):
            preds.append({
                "boxes": outputs[i]["boxes"].cpu(),
                "scores": outputs[i]["scores"].cpu(),
                "labels": outputs[i]["labels"].cpu()
            })

            gts.append({
                "boxes": targets[i]["boxes"].cpu(),
                "labels": targets[i]["labels"].cpu()
            })

        metric.update(preds, gts)


C:\Users\menci\anaconda3\envs\SSN\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


In [16]:
results = metric.compute()


In [17]:
ap50 = results["map_50"]


In [18]:
metric = MeanAveragePrecision()  # usa todos los IoU


In [19]:
ap5095 = results["map"]


In [20]:
recall = results["mar_100"]
precision = results["map_50"] / recall if recall > 0 else 0
f1 = 2 * precision * recall / (precision + recall + 1e-6)


In [21]:
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"AP@0.5:    {ap50:.4f}")
print(f"AP@0.5:0.95: {ap5095:.4f}")


Precision: 0.0006
Recall:    0.0021
F1-score:  0.0009
AP@0.5:    0.0000
AP@0.5:0.95: 0.0000
